In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [ ]:
import csv
import json
from models.llm_ner import get_entities_from_llm
from models.llm_text_embeddings import get_embeddings
from models.metric import CDE, exhaustive_CDE, EF, CDEF
from models.traditional_metrics import precision, recall, f1_score

In [3]:
prompt_1 = """
# Instruction
Analyze the sentence and extract from it all entities of types {types}. 
Assign to them types from the below list.
Keep in mind, that entities could be discontinuous, long and descriptive.

## Sentence
{sentence}

## Types
{types}
"""

prompt_2 = """
# Instruction
Analyze the sentence and extract from it all entities of types {types}. 
Assign to them types from the below list.
Keep in mind, that entities could be discontinuous, long and descriptive.
Base on below examples.

## Sentence
{sentence}

## Types
{types}

## Examples
### Example-1 discontinuous entity
- Sentence: "the muscle and joints in my angles hurt",
- Entities: [['the muscle in the angles hurt', 'Symptom'], ['joints in the angles hurt', 'Symptom']
### Example-2 discontinuous entity
 - Sentence: "lightheadedness and blurry vision when standing"
 - Entities: [['lightheadedness when standing', 'Symptom'], ['blurry vision when standing', 'Symptom']]
### Example-3 long, descriptive span
 - Sentence: "severe pain in the muscles in the shoulder area"
 - Entities: [['severe pain in the muscles in the shoulder area', 'Symptom']]
### Example-4 long, descriptive span
 - Sentence: "I did take my blood pressure today though, and it was slightly elevated from the norm."
 - Entities: [['blood pressure slightly elevated', 'Symptom']]
"""

In [4]:
def parse_response(response:str):
    response = json.loads(response)
    
    entities = []

    for e in response['entities']:
        entity = e['entity']
        type = e['type']
        entities.append([entity, type])

    return entities

In [5]:
def get_tp_fp_fn(gold_entities, generated_entities):
    tp = [0]*len(generated_entities)
    fp = [1]*len(generated_entities)
    fn = [1]*len(gold_entities)

    for i, gen_e in enumerate(generated_entities):
        for j, gold_e in enumerate(gold_entities):
            if gen_e == gold_e:
                fn[j]=0
                tp[i]=1
                fp[i]=0
    
    return sum(tp), sum(fp), sum(fn)


In [6]:
def run_test_on_use_case(prompt, types, sentence, gold_entities, verbose=True):
    
    response = get_entities_from_llm(sentence, types, prompt=prompt)
    generated_entities = parse_response(response)

    if verbose:
        print(f'{"SENTENCE:":20s} {sentence}')
        print(f'{"GOLD_ENTITIES:":20s} {gold_entities}')
        print(f'{"GENERATED_ENTITIES:":20s} {generated_entities}')

    tp, fp, fn = get_tp_fp_fn(gold_entities, generated_entities)

    gold_embeddings = get_embeddings(gold_entities)
    generated_embeddings = get_embeddings(generated_entities)
    
    cde=CDE(gold_embeddings, generated_embeddings)
    exh_cde=exhaustive_CDE(gold_embeddings, generated_embeddings)
    ef=EF(gold_embeddings, generated_embeddings)
    cdef_05=CDEF(gold_embeddings, generated_embeddings, beta=0.5)
    cdef_1=CDEF(gold_embeddings, generated_embeddings, beta=1)
    cdef_15=CDEF(gold_embeddings, generated_embeddings, beta=1.5)
    prec=precision(tp,fp,fn)
    rec=recall(tp,fp,fn)
    f1=f1_score(tp,fp,fn)

    if verbose:
        print(f'{"CDE:":10s}{cde}')
        print(f'{"Exh_CDE":10s}{exh_cde}')
        print(f'{"EF:":10s}{ef}')
        print(f'{"CDEF-0.5":10s}{cdef_05}')
        print(f'{"CDEF-1.0:":10s}{cdef_1}')
        print(f'{"PREC:":10s}{prec}')
        print(f'{"REC:":10s}{rec}')
        print(f'{"F1:":10s}{f1}')

    return cde, exh_cde, ef, cdef_05, cdef_1, cdef_15, prec, rec, f1

In [ ]:
def write_to_csv():
    
import csv
with open('eggs.csv', 'w', newline='') as csvfile:
    spamwriter = csv.writer(csvfile, delimiter=' ',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)
    spamwriter.writerow(['Spam'] * 5 + ['Baked Beans'])
    spamwriter.writerow(['Spam', 'Lovely Spam', 'Wonderful Spam'])

# Use Cases
Three types of entities:
- Symptom,
- Drug,
- Disease.

In [7]:
types = ['Symptom', 'Drug', 'Disease']

## 1. Discontinuous Entities

### UC1.1 
**The patient reported pain in the lower back and occasionally in the right leg.**
- pain in the lower back; Symptom
- pain in the right leg; Symptom

In [10]:
sentence = "The patient reported pain in the lower back and occasionally in the right leg."
gold_entities = [
    ['pain in the lower back', 'Symptom'],
    ['pain in the right leg', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            The patient reported pain in the lower back and occasionally in the right leg.
GOLD_ENTITIES:       [['pain in the lower back', 'Symptom'], ['pain in the right leg', 'Symptom']]
GENERATED_ENTITIES:  [['pain', 'Symptom'], ['lower back', 'Symptom'], ['right leg', 'Symptom']]
CDE:      0.1215967271289205
Exh_CDE   0.1215967271289205
EF:       0.19999999999999996
CDEF-0.5  0.9076162206432932
CDEF-1.0: 0.8640301313059162
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            The patient reported pain in the lower back and occasionally in the right leg.
GOLD_ENTITIES:       [['pain in the lower back', 'Symptom'], ['pain in the right leg', 'Symptom']]
GENERATED_ENTITIES:  [['pain in the lower back', 'Symptom'], ['occasionally in the right leg pain', 'Symptom']]
CDE:      0.03107010501371621
Exh_CDE   0.03107010501371621
EF:       0.0
CDEF-0.5  0.9875332235892872
CDEF-1.0: 0.9921716669641948
PREC:     0.5
REC:      0.5
F1:       0.5


(0.03107010501371621,
 0.03107010501371621,
 0.0,
 0.9875332235892872,
 0.9921716669641948,
 0.9951680156770027,
 0.5,
 0.5,
 0.5)

### UC1.2
**The patient experienced shortness of breath on exertion and sometimes at rest.**
- shortness of breath on exertion; Symptom
- shortness of breath at rest; Symptom

In [9]:
sentence = "The patient experienced shortness of breath on exertion and sometimes at rest."
gold_entities = [
    ['shortness of breath on exertion', 'Symptom'],
    ['shortness of breath at rest', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            The patient experienced shortness of breath on exertion and sometimes at rest.
GOLD_ENTITIES:       [['shortness of breath on exertion', 'Symptom'], ['shortness of breath at rest', 'Symptom']]
GENERATED_ENTITIES:  [['shortness of breath', 'Symptom']]
CDE:      0.11803836561393666
Exh_CDE   0.11803836561393666
EF:       -0.33333333333333337
CDEF-0.5  0.8694315999580503
CDEF-1.0: 0.7804205226499786
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            The patient experienced shortness of breath on exertion and sometimes at rest.
GOLD_ENTITIES:       [['shortness of breath on exertion', 'Symptom'], ['shortness of breath at rest', 'Symptom']]
GENERATED_ENTITIES:  [['shortness of breath', 'Symptom'], ['exertion', 'Disease'], ['at rest', 'Disease']]
CDE:      1.0590191828069684
Exh_CDE   1.0
EF:       0.19999999999999996
CDEF-0.5  0.5127275717649044
CDEF-1.0: 0.5925150230657867
PREC:     0.0
REC:      0.0
F1:       0


(1.0590191828069684,
 1.0,
 0.19999999999999996,
 0.5127275717649044,
 0.5925150230657867,
 0.6581689507403955,
 0.0,
 0.0,
 0)

### UC1.3
**Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.**
- muscle pain; Symptom
- joint pain; Symptom

In [10]:
sentence = "Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better."
gold_entities = [
    ['muscle pain', 'Symptom'],
    ['joint pain', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.
GOLD_ENTITIES:       [['muscle pain', 'Symptom'], ['joint pain', 'Symptom']]
GENERATED_ENTITIES:  [['muscle/joint pain', 'Symptom']]
CDE:      0.10130145729433637
Exh_CDE   0.10130145729433637
EF:       -0.33333333333333337
CDEF-0.5  0.8751337453753893
CDEF-1.0: 0.7832837527714838
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.
GOLD_ENTITIES:       [['muscle pain', 'Symptom'], ['joint pain', 'Symptom']]
GENERATED_ENTITIES:  [['muscle/joint pain', 'Symptom'], ['sleeping much better', 'Symptom']]
CDE:      0.3650501006448187
Exh_CDE   0.3650501006448187
EF:       0.0
CDEF-0.5  0.8484475355002107
CDEF-1.0: 0.8995721782273872
PREC:     0.0
REC:      0.0
F1:       0


(0.3650501006448187,
 0.3650501006448187,
 0.0,
 0.8484475355002107,
 0.8995721782273872,
 0.93571519308951,
 0.0,
 0.0,
 0)

### UC1.4
**Menstrual cramps present with or without vaginal bleeding.**
- menstrual cramps with vaginal bleeding; Symptom
- menstrual cramps without vaginal bleeding; Symptom

In [11]:
sentence = "Menstrual cramps present with or without vaginal bleeding."
gold_entities = [
    ['menstrual cramps with vaginal bleeding', 'Symptom'],
    ['menstrual cramps without vaginal bleeding', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            Menstrual cramps present with or without vaginal bleeding.
GOLD_ENTITIES:       [['menstrual cramps with vaginal bleeding', 'Symptom'], ['menstrual cramps without vaginal bleeding', 'Symptom']]
GENERATED_ENTITIES:  [['menstrual cramps', 'Symptom'], ['vaginal bleeding', 'Symptom']]
CDE:      0.14882194156765166
Exh_CDE   0.14882194156765166
EF:       0.0
CDEF-0.5  0.9395719209682687
CDEF-1.0: 0.9613567746519021
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            Menstrual cramps present with or without vaginal bleeding.
GOLD_ENTITIES:       [['menstrual cramps with vaginal bleeding', 'Symptom'], ['menstrual cramps without vaginal bleeding', 'Symptom']]
GENERATED_ENTITIES:  [['Menstrual cramps', 'Symptom'], ['vaginal bleeding', 'Symptom']]
CDE:      0.14882194156765166
Exh_CDE   0.14882194156765166
EF:       0.0
CDEF-0.5  0.9395719209682687
CDEF-1.0: 0.9613567746519021
PREC:     0.0
REC:      0.0
F1:       0


(0.14882194156765166,
 0.14882194156765166,
 0.0,
 0.9395719209682687,
 0.9613567746519021,
 0.9758607777062375,
 0.0,
 0.0,
 0)

### UC1.5
**After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.**
- abdominal gas; Symptom
- abdominal cramps; Symptom
- abdominal pain; Symptom

In [12]:
sentence = "After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day."
gold_entities = [
    ['abdominal gas', 'Symptom'],
    ['abdominal cramps', 'Symptom'],
    ['abdominal pain', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.
GOLD_ENTITIES:       [['abdominal gas', 'Symptom'], ['abdominal cramps', 'Symptom'], ['abdominal pain', 'Symptom']]
GENERATED_ENTITIES:  [['abdominal gas', 'Symptom'], ['cramps', 'Symptom'], ['pain', 'Symptom']]
CDE:      0.1263790903944034
Exh_CDE   0.1263790903944034
EF:       0.0
CDEF-0.5  0.948801319576102
CDEF-1.0: 0.9673744299342727
PREC:     0.3333333333333333
REC:      0.3333333333333333
F1:       0.3333333333333333
SENTENCE:            After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.
GOLD_ENTITIES:       [['abdominal gas', 'Symptom'], ['abdominal cramps', 'Symptom'], ['abdominal pain', 'Symptom']]
GENERATED_ENTITIES:  [['abdominal gas', 'Symptom'], ['cramps', 'Symptom'], ['pain', 'Symptom']]
CDE:      0.1263790903944034
Exh_CDE   0.1263790903944034
EF:       0.

(0.1263790903944034,
 0.1263790903944034,
 0.0,
 0.948801319576102,
 0.9673744299342727,
 0.9796675889981112,
 0.3333333333333333,
 0.3333333333333333,
 0.3333333333333333)

## 2. Long, Descriptive Entities

### UC2.1
**Without this drug I was severly restricted and could only walk less than 100 meters.**
- could only walk less than 100 meters; Symptom

In [13]:
sentence = "Without this drug I was severly restricted and could only walk less than 100 meters."
gold_entities = [
    ['could only walk less than 100 meters', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            Without this drug I was severly restricted and could only walk less than 100 meters.
GOLD_ENTITIES:       [['could only walk less than 100 meters', 'Symptom']]
GENERATED_ENTITIES:  [['this drug', 'Drug']]
CDE:      2.0
Exh_CDE   2.0
EF:       0.0
CDEF-0.5  0.0
CDEF-1.0: 0.0
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            Without this drug I was severly restricted and could only walk less than 100 meters.
GOLD_ENTITIES:       [['could only walk less than 100 meters', 'Symptom']]
GENERATED_ENTITIES:  [['this drug', 'Drug']]
CDE:      2.0
Exh_CDE   2.0
EF:       0.0
CDEF-0.5  0.0
CDEF-1.0: 0.0
PREC:     0.0
REC:      0.0
F1:       0


(2.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0)

### UC2.2
**Recently I've been feeling like my stomach is full and empty at the same time hard to explain.**
- feeling stomach is full and empty at the same time; Symptom

In [14]:
sentence = "Recently I've been feeling like my stomach is full and empty at the same time hard to explain."
gold_entities = [
    ['feeling stomach is full and empty at the same time', 'Symptom']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            Recently I've been feeling like my stomach is full and empty at the same time hard to explain.
GOLD_ENTITIES:       [['feeling stomach is full and empty at the same time', 'Symptom']]
GENERATED_ENTITIES:  [['stomach', 'Disease'], ['full', 'Symptom'], ['empty', 'Symptom']]
CDE:      0.3367578766208573
Exh_CDE   0.3367578766208573
EF:       0.5
CDEF-0.5  0.7342270059904835
CDEF-1.0: 0.6245178043627546
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            Recently I've been feeling like my stomach is full and empty at the same time hard to explain.
GOLD_ENTITIES:       [['feeling stomach is full and empty at the same time', 'Symptom']]
GENERATED_ENTITIES:  [['feeling like stomach is full and empty at the same time', 'Symptom']]
CDE:      0.014641006109458288
Exh_CDE   0.014641006109458288
EF:       0.0
CDEF-0.5  0.9941350106216846
CDEF-1.0: 0.9963263018132362
PREC:     0.0
REC:      0.0
F1:       0


(0.014641006109458288,
 0.014641006109458288,
 0.0,
 0.9941350106216846,
 0.9963263018132362,
 0.9977360638011035,
 0.0,
 0.0,
 0)

### UC2.3
**The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.**
- lung infection of the lower part of right lung; Disease

In [15]:
sentence = "The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung."
gold_entities = [
    ['lung infection of the lower part of right lung', 'Disease']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.
GOLD_ENTITIES:       [['lung infection of the lower part of right lung', 'Disease']]
GENERATED_ENTITIES:  [['lung infection', 'Disease']]
CDE:      0.18487491599217287
Exh_CDE   0.18487491599217287
EF:       0.0
CDEF-0.5  0.924657132982076
CDEF-1.0: 0.9515415846345043
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.
GOLD_ENTITIES:       [['lung infection of the lower part of right lung', 'Disease']]
GENERATED_ENTITIES:  [['lung infection', 'Disease']]
CDE:      0.18487491599217287
Exh_CDE   0.18487491599217287
EF:       0.0
CDEF-0.5  0.924657132982076
CDEF-1.0: 0.9515415846345043
PREC:     0.0
REC:      0.0
F1:       0


(0.18487491599217287,
 0.18487491599217287,
 0.0,
 0.924657132982076,
 0.9515415846345043,
 0.9696130899642383,
 0.0,
 0.0,
 0)

### UC2.4
**The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.**
- strong intravenous medicine; Drug
- swelling in the brain; Disease
- immune system attacking itself; Disease

In [16]:
sentence = "The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself."
gold_entities = [
    ['strong intravenous medicine', 'Drug'],
    ['swelling in the brain', 'Disease'],
    ['immune system attacking itself', 'Disease']
]

run_test_on_use_case(prompt_1, types, sentence, gold_entities)
run_test_on_use_case(prompt_2, types, sentence, gold_entities)

SENTENCE:            The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.
GOLD_ENTITIES:       [['strong intravenous medicine', 'Drug'], ['swelling in the brain', 'Disease'], ['immune system attacking itself', 'Disease']]
GENERATED_ENTITIES:  [['swelling', 'Symptom'], ['strong medicine', 'Drug'], ['the immune system attacking itself', 'Disease']]
CDE:      0.7233145943836132
Exh_CDE   0.6666666666666666
EF:       0.0
CDEF-0.5  0.6881150700892814
CDEF-1.0: 0.7792541837724735
PREC:     0.0
REC:      0.0
F1:       0
SENTENCE:            The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.
GOLD_ENTITIES:       [['strong intravenous medicine', 'Drug'], ['swelling in the brain', 'Disease'], ['immune system attacking itself', 'Disease']]
GENERATED_ENTITIES:  [['swelling in the brain', 'Symptom'], ['immune system attacking itself', 'Disease']]
CD

(0.9999999999999999,
 0.9999999999999999,
 -0.19999999999999996,
 0.5405405405405405,
 0.6153846153846154,
 0.6753246753246753,
 0.5,
 0.3333333333333333,
 0.4)